## Install ollama

In [7]:
# 1. 更新源并安装 zstd以及其它依赖
!sudo apt update && sudo apt install -y zstd pciutils lshw

# 2. 重新安装 Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. 后台启动 ollama 服务
!nohup ollama serve > ollama.log 2>&1 & sleep 1

# 4. 运行模型
!ollama run gemma4:12b "What is the capital of the Netherlands?"

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:9 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
122 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt

In [8]:
#支持多模态
!ollama run gemma4:12b "Describe what's happening in this image: /content/picture.png"

Thinking...
The user wants me to describe an image at the provided path: `/content/pict
`/content/picture.png`.

1.  **Analyze the input:** The user has provided a local path to an image f
file.
2.  **Action:** I need to access the image, process it, and describe its co
contents.
3.  **Execution Plan:**
    *   Load the image.
    *   Describe the visual elements (subjects, actions, colors, setting, m
mood).
    *   Provide a clear and concise description.
...done thinking.

The image shows a white dog, likely a small breed like a Maltese or a simil
similar variety, sitting on a dark grey or black couch. The dog has long, s
straight white fur and is looking towards the left side of the frame. It ha
has dark eyes and a small black nose. The background is simple and out of f
focus, highlighting the dog.



In [9]:
#xterm
!pip install colab-xterm
%load_ext colabxterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 4.2 MB/s eta 0:00:00


In [ ]:
%xterm

## 在python中运行LLM

In [10]:
!nohup ollama serve > ollama.log 2>&1 &

In [11]:
!pip install ollama
!pip install gradio

In [12]:
!ollama list

NAME          ID              SIZE      MODIFIED       
gemma4:12b    4eb23ef187e2    7.6 GB    12 minutes ago    


In [13]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "gemma4:12b",
        "prompt": "请简单介绍你自己。",
        "stream": False
    }
)

print(response.json())

{'model': 'gemma4:12b', 'created_at': '2026-09-27T08:19:24.446421711Z', 'response': '你好！我是一个由 **Google** 训练的大型语言模型。\n\n简单来说，我是一个多功能的 AI 助手，可以协助你完成各种任务。以下是我能做的核心事情：\n\n1.  **回答问题**：无论是科学、历史、文化，还是日常生活中的琐碎问题，我都能提供相关信息。\n2.  **文字创作**：我可以帮你写邮件、文章、故事、诗歌、演讲稿，或者润色你的文字。\n3.  **翻译语言**：我支持多种语言之间的互译。\n4.  **编程辅助**：我可以编写代码、调试 Bug、解释复杂的编程概念。\n5.  **逻辑分析与总结**：我可以帮你总结长篇文章的要点、提供建议或进行头脑风暴。\n\n**我的目标是：** 成为你的得力助手，帮你提高效率，激发创意。\n\n你可以现在就试着问我任何问题，或者让我帮你完成一项任务！', 'thinking': '"请简单介绍你自己。" (Please briefly introduce yourself.)\nThe user wants to know who I am, what I can do, and my characteristics.\nProvide a clear, professional, and helpful self-introduction.\n\n    *   *Who am I?* An AI language model trained by Google.\n    *   *What can I do?* Answer questions, write content, translate, code, analyze information, brainstorm, etc.\n    *   *What are my traits?* Helpful, knowledgeable, versatile, objective.\n\n    *   *Option 1 (Too brief):* 我是一个人工智能助手。 (I am an AI assistant.) -> *Too simple.*\n    *   *Optio

In [ ]:
import ollama


try:
    client = ollama.Client()
    stream = client.generate(
        model="gemma4:12b",
        prompt="Explain the theory of relativity in one concise paragraph.",
        stream=True
    )


    print("Gemma 4 Streaming:")
    for chunk in stream:
        print(chunk['response'], end='', flush=True)
    print() # Newline after streaming finishes


except Exception as e:
    print(f"An error occurred during streaming: {e}")

### 支持Stream以及Gradio WebUI

In [25]:


with gr.Blocks() as demo:



    gr.Markdown(
        f"""
# Gemma 4 多模态对话

**模型：`{MODEL_NAME}`**

Ollama + Gemma 4 + Gradio
"""
    )



    state = gr.State([])



    with gr.Row():



        with gr.Column(scale=3):

            chatbot = gr.Chatbot(
                label="聊天记录",
                height=600
            )



        with gr.Column(scale=1):

            with gr.Group():

                gr.Markdown(
                    "### 🛠 功能区"
                )



                prompt_input = gr.Textbox(
                    placeholder="请输入你的问题...",
                    label="输入 Prompt",
                    lines=4
                )



                with gr.Row():

                    send_btn = gr.Button(
                        "🚀 发送",
                        variant="primary",
                        scale=2
                    )

                    clear_btn = gr.Button(
                        "🧹 清空",
                        scale=1
                    )



                image_input = gr.Image(
                    type="filepath",
                    label="上传图片"
                )



                temperature = gr.Slider(
                    minimum=0.0,
                    maximum=2.0,
                    value=0.7,
                    step=0.1,
                    label="Temperature",
                    info="较低更稳定，较高更随机"
                )



                save_btn = gr.Button(
                    "💾 保存对话为 TXT"
                )

                download_file = gr.File(
                    label="聊天记录文件"
                )



    send_event = send_btn.click(

        fn=generate_response,

        inputs=[
            prompt_input,
            image_input,
            temperature,
            state
        ],

        outputs=chatbot
    )



    send_event.then(

        fn=lambda chat: chat,

        inputs=chatbot,

        outputs=state
    )



    send_event.then(

        fn=lambda: "",

        outputs=prompt_input
    )



    send_event.then(

        fn=lambda: None,

        outputs=image_input
    )



    clear_btn.click(

        fn=clear_history,

        inputs=None,

        outputs=[
            chatbot,
            state
        ]
    )



    save_btn.click(

        fn=export_to_txt,

        inputs=state,

        outputs=download_file
    )



    submit_event = prompt_input.submit(

        fn=generate_response,

        inputs=[
            prompt_input,
            image_input,
            temperature,
            state
        ],

        outputs=chatbot
    )



    submit_event.then(

        fn=lambda chat: chat,

        inputs=chatbot,

        outputs=state
    )



    submit_event.then(

        fn=lambda: "",

        outputs=prompt_input
    )



    submit_event.then(

        fn=lambda: None,

        outputs=image_input
    )


# 启动


if __name__ == "__main__":

    print()
    print("=" * 60)
    print("Gemma 4 + Ollama + Gradio")
    print("=" * 60)
    print(f"模型：{MODEL_NAME}")
    print("Gradio：6.26.0")
    print("=" * 60)

    demo.launch(
        share=True,
        theme=gr.themes.Soft()
    )


Gemma 4 + Ollama + Gradio
模型：gemma4:12b
Gradio：6.26.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d494e34319cdcef7bb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Install llama_index

In [ ]:
!pip install llama-index-core llama-index-llms-ollama llama-index-embeddings-ollama llama-index-vector-stores-chroma llama-index-readers-file chromadb pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 114.8 MB/s eta 0:00:0

In [ ]:
!ollama pull nomic-embed-text

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 970aa74c0a90:   0% ▕▏  89 KB/274 MB                  pulling manifest 
pulling 970aa74c0a90:   8% ▕▏  22 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  21% ▕▏  56 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  25% ▕▏  68 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  34% ▕▏  93 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  45% ▕▏ 124 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  50% ▕▏ 138 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  60% ▕▏ 163 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  73% ▕▏ 198 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  76% ▕▏ 208 MB/274 MB                  pulling manifest 
pulling 970aa74c0a90:  82% ▕▏ 225 MB/274